# 30.07 - Object detection foundations

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** Detection data audit + box utility notebook.

You will inspect a tiny YOLO-style dataset, convert box formats, calculate Intersection over Union (IoU), apply non-maximum suppression (NMS), and visualize an audit sample. Everything is deterministic and local.

## Core Ideas

- **Classification** predicts what is in an image; **detection** predicts what and where. A detection usually contains a box, class label, and confidence score.
- `xyxy = [x_min, y_min, x_max, y_max]` is convenient for overlap calculations. `xywh = [x_center, y_center, width, height]` is common in YOLO labels. YOLO text labels normalize coordinates to `[0, 1]`.
- IoU is intersection area divided by union area. It is 0 for non-overlapping boxes and 1 for identical boxes.
- NMS keeps high-confidence boxes and suppresses lower-confidence boxes that overlap too much. In production, use the optimized `torchvision.ops.nms` implementation.
- Precision-recall evaluates confidence-ranked predictions. AP summarizes one class's precision-recall curve; mAP averages AP across classes and, depending on the benchmark, IoU thresholds.

Box convention in this notebook: coordinates are continuous `float32` values, `x_max` and `y_max` are the far edges, and valid boxes have positive width and height.

## Competition-Library Model Check

This foundations lesson does not need a full detector. For competition code, Torchvision already provides optimized `box_iou`, `generalized_box_iou`, `nms`, and `batched_nms`; the manual IoU exercise remains here only because box overlap is the explicit learning objective. Later detector baselines should use official Torchvision architectures rather than rebuilding them.

In [ ]:
import os
import csv
import numpy as np
import torch
from PIL import Image, ImageDraw
from torchvision.ops import nms
import matplotlib.pyplot as plt

SEED = 30
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_ROOT = "_day30_detection_data"
IMAGE_DIR = os.path.join(DATA_ROOT, "images")
LABEL_DIR = os.path.join(DATA_ROOT, "labels")
os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(LABEL_DIR, exist_ok=True)

CLASS_NAMES = {0: "circle", 1: "rectangle"}

## Prepared Detection Data

The provided cell writes six `96 x 96` RGB images, matching YOLO label files, and an audit CSV. Each YOLO row is `class_id x_center y_center width height`, normalized by image size. Data generation is provided because the learning target is detection tooling, not fixture construction.

In [ ]:
records = []
for image_index in range(6):
    canvas = Image.new("RGB", (96, 96), color=(238, 241, 245))
    painter = ImageDraw.Draw(canvas)
    boxes = []
    x1 = 8 + 5 * image_index
    y1 = 10 + 3 * image_index
    x2 = x1 + 24
    y2 = y1 + 22
    painter.ellipse((x1, y1, x2, y2), fill=(225, 90, 85), outline=(120, 20, 20), width=2)
    boxes.append((0, x1, y1, x2, y2))
    if image_index % 2 == 0:
        rx1, ry1 = 52 - image_index, 48 + image_index
        rx2, ry2 = rx1 + 29, ry1 + 20
        painter.rectangle((rx1, ry1, rx2, ry2), fill=(75, 145, 225), outline=(15, 55, 125), width=2)
        boxes.append((1, rx1, ry1, rx2, ry2))

    image_name = f"image_{image_index:02d}.png"
    label_name = f"image_{image_index:02d}.txt"
    canvas.save(os.path.join(IMAGE_DIR, image_name))
    with open(os.path.join(LABEL_DIR, label_name), "w", encoding="utf-8") as handle:
        for class_id, bx1, by1, bx2, by2 in boxes:
            xc = ((bx1 + bx2) / 2.0) / 96.0
            yc = ((by1 + by2) / 2.0) / 96.0
            width = (bx2 - bx1) / 96.0
            height = (by2 - by1) / 96.0
            handle.write(f"{class_id} {xc:.6f} {yc:.6f} {width:.6f} {height:.6f}\n")
    records.append({"image_path": os.path.join("images", image_name), "label_path": os.path.join("labels", label_name)})

with open(os.path.join(DATA_ROOT, "audit.csv"), "w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["image_path", "label_path"])
    writer.writeheader()
    writer.writerows(records)

print("prepared images:", len(records))
print("first record:", records[0])

## Exercise 30-A: Load normalized YOLO annotations

Read one label file, validate each row, and return tensors. Reject class IDs outside `class_names` and normalized coordinates outside `[0, 1]`.

**Return structure — `load_yolo_labels`:** A dictionary with exactly two keys: `labels`, a CPU `torch.int64` tensor of shape `[N]`; and `boxes_xywhn`, a CPU `torch.float32` tensor of shape `[N, 4]`. Here `N` is the number of objects and each row is normalized `[x_center, y_center, width, height]`.

In [ ]:
# TODO 30-A
def load_yolo_labels(label_path, class_names):
    # Parse whitespace-separated rows and validate all values.
    raise NotImplementedError("Complete Exercise 30-A")


# Smoke check: run this after implementing the function above.
sample_target = load_yolo_labels(os.path.join(LABEL_DIR, "image_00.txt"), CLASS_NAMES)
print("labels:", sample_target["labels"].tolist())
print("normalized boxes:", sample_target["boxes_xywhn"])

## Exercise 30-B: Convert box formats

Convert normalized YOLO `xywh` boxes to pixel `xyxy`, then convert them back to pixel `xywh`. Clamp `xyxy` coordinates to the image boundary.

**Return structure — `xywhn_to_xyxy`:** A CPU `torch.float32` tensor `[N, 4]` containing pixel `[x_min, y_min, x_max, y_max]`.

**Return structure — `xyxy_to_xywh`:** A tensor `[N, 4]` with the same dtype and device as its input, containing pixel `[x_center, y_center, width, height]`.

In [ ]:
# TODO 30-B
def xywhn_to_xyxy(boxes_xywhn, image_width, image_height):
    raise NotImplementedError("Complete Exercise 30-B")


def xyxy_to_xywh(boxes_xyxy):
    raise NotImplementedError("Complete Exercise 30-B")


# Smoke check: run this after implementing both functions above.
sample_xyxy = xywhn_to_xyxy(sample_target["boxes_xywhn"], 96, 96)
sample_xywh = xyxy_to_xywh(sample_xyxy)
print("pixel xyxy:", sample_xyxy)
print("pixel xywh:", sample_xywh)

## Exercise 30-C: Calculate pairwise IoU

Implement the IoU matrix using tensor operations. Protect the denominator with a small epsilon and handle empty inputs without changing the output shape.

**Return structure — `pairwise_iou`:** A tensor on the input device with shape `[N, M]` and floating dtype. Element `[i, j]` is the IoU between `boxes_a[i]` and `boxes_b[j]`, bounded by `[0, 1]`.

In [ ]:
# TODO 30-C
def pairwise_iou(boxes_a, boxes_b, eps=1e-7):
    raise NotImplementedError("Complete Exercise 30-C")


# Smoke check: run this after implementing the function above.
iou_demo = pairwise_iou(
    torch.tensor([[0.0, 0.0, 10.0, 10.0]]),
    torch.tensor([[0.0, 0.0, 10.0, 10.0], [20.0, 20.0, 30.0, 30.0]]),
)
print("IoU demo:", iou_demo)

## Exercise 30-D: Apply class-aware NMS

Use `torchvision.ops.nms` independently for each class so boxes from different classes do not suppress one another. Return kept indices ordered by descending score.

**Return structure — `class_aware_nms`:** A CPU `torch.int64` tensor `[K]`, where `0 <= K <= N`. Values are unique indices into the original boxes and are ordered by descending original score.

In [ ]:
# TODO 30-D
def class_aware_nms(boxes_xyxy, scores, labels, iou_threshold=0.5):
    raise NotImplementedError("Complete Exercise 30-D")


# Smoke check: run this after implementing the function above.
prediction_boxes = torch.tensor([[8, 10, 32, 32], [10, 11, 33, 33], [52, 48, 81, 68], [53, 49, 80, 69]], dtype=torch.float32)
prediction_scores = torch.tensor([0.95, 0.72, 0.91, 0.60], dtype=torch.float32)
prediction_labels = torch.tensor([0, 0, 1, 1], dtype=torch.int64)
kept_indices = class_aware_nms(prediction_boxes, prediction_scores, prediction_labels, 0.5)
print("kept indices:", kept_indices.tolist())

## Exercise 30-E: Draw an annotation audit

Draw labeled boxes on a copy of an image. This visual check catches swapped axes, incorrect scaling, and class-mapping errors that shape tests cannot.

**Return structure — `draw_box_audit`:** A new `PIL.Image.Image` in RGB mode with the same `(width, height)` as the input. The input image is not modified.

In [ ]:
# TODO 30-E
def draw_box_audit(image, boxes_xyxy, labels, class_names):
    raise NotImplementedError("Complete Exercise 30-E")


# Smoke check: run this after implementing the function above.
source_image = Image.open(os.path.join(IMAGE_DIR, "image_00.png")).convert("RGB")
audit_image = draw_box_audit(source_image, sample_xyxy, sample_target["labels"], CLASS_NAMES)
plt.figure(figsize=(4, 4))
plt.imshow(audit_image)
plt.axis("off")
plt.show()

## Test Cases

Run this cell after completing all TODOs.

**Return structure — `run_day30_tests`:** Returns `None`. Success is communicated by assertions completing and the exact printed message `Day 30 tests passed`.

In [ ]:
def run_day30_tests():
    assert os.path.isfile(os.path.join(DATA_ROOT, "audit.csv"))
    assert len(os.listdir(IMAGE_DIR)) == 6
    target = load_yolo_labels(os.path.join(LABEL_DIR, "image_00.txt"), CLASS_NAMES)
    assert set(target) == {"labels", "boxes_xywhn"}
    assert target["labels"].dtype == torch.int64 and target["labels"].shape == (2,)
    assert target["boxes_xywhn"].dtype == torch.float32 and target["boxes_xywhn"].shape == (2, 4)
    boxes = xywhn_to_xyxy(target["boxes_xywhn"], 96, 96)
    assert boxes.shape == (2, 4) and torch.all(boxes >= 0) and torch.all(boxes <= 96)
    reconstructed = xyxy_to_xywh(boxes)
    expected_pixel_xywh = target["boxes_xywhn"] * torch.tensor([96.0, 96.0, 96.0, 96.0])
    assert torch.allclose(reconstructed, expected_pixel_xywh, atol=1e-3)
    identity_iou = pairwise_iou(boxes, boxes)
    assert identity_iou.shape == (2, 2)
    assert torch.allclose(torch.diag(identity_iou), torch.ones(2), atol=1e-5)
    empty_iou = pairwise_iou(torch.empty((0, 4)), boxes)
    assert empty_iou.shape == (0, 2)
    kept = class_aware_nms(prediction_boxes, prediction_scores, prediction_labels, 0.5)
    assert kept.dtype == torch.int64 and kept.tolist() == [0, 2]
    rendered = draw_box_audit(source_image, boxes, target["labels"], CLASS_NAMES)
    assert isinstance(rendered, Image.Image) and rendered.mode == "RGB" and rendered.size == source_image.size
    print("Day 30 tests passed")


run_day30_tests()

## Day 30 Checklist

- [ ] I can explain why detection outputs need boxes, labels, and confidence scores.
- [ ] I can convert normalized `xywh` boxes to pixel `xyxy` boxes and back.
- [ ] I tested identical, disjoint, and empty-box IoU cases.
- [ ] I used optimized, class-aware NMS and can explain its threshold.
- [ ] I visually audited annotations instead of trusting numeric checks alone.
- [ ] I can distinguish AP@50 from COCO-style mAP@50:95.